# Random ADF STEM Image Grid

This notebook samples standalone ADF/HAADF STEM image records from the rebuilt `image_sequence` table and plots an 8 x 5 image grid. The `image_sequence` table already excludes ADF SI-raster child images, so the sample is drawn from standalone STEM image records rather than secondary raster images collected during spectrum imaging.


In [ ]:
from pathlib import Path
import os
import random
import sqlite3

os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib-cache").resolve()))

import matplotlib.pyplot as plt
import numpy as np
from rsciio.digitalmicrograph import file_reader

ARCHIVE_ROOT = Path(os.environ.get("STEM_BANK_26_ARCHIVE_ROOT", "STEM-Bank-26 Archive Sampler"))
TRAJECTORY_DB = ARCHIVE_ROOT / "_archive_inventory" / "stem_bank_26_trajectories_by_acq_date_sample.sqlite"

N_ROWS = 8
N_COLS = 5
N_IMAGES = N_COLS * N_ROWS
RANDOM_SEED = 20260829

# ADF SI-raster child images are not in image_sequence. Set this to True if
# SI survey images should also be excluded from the visual sample.
EXCLUDE_SI_SURVEYS = False

FIGURE_PATH = Path("figures/random_adf_stem_grid.png")


In [ ]:
def fetch_adf_candidates(db_path: Path) -> list[sqlite3.Row]:
    where = ["(lower(title) LIKE '%adf%' OR lower(path) LIKE '%adf%')"]
    if EXCLUDE_SI_SURVEYS:
        where.append("lower(title) NOT LIKE '%survey%'")
        where.append("lower(path) NOT LIKE '%survey%'")

    query = f"""
        SELECT
            image_id,
            session_key,
            trajectory_key,
            path,
            title,
            acquisition_date,
            acquisition_time,
            data_shape,
            formatted_magnification_text,
            formatted_voltage_text,
            field_of_view_um_number,
            stage_x_number,
            stage_y_number
        FROM image_sequence
        WHERE {' AND '.join(where)}
        ORDER BY image_id
    """

    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        return list(conn.execute(query))


candidates = fetch_adf_candidates(TRAJECTORY_DB)
rng = random.Random(RANDOM_SEED)
sampled_rows = rng.sample(candidates, min(N_IMAGES, len(candidates)))

print(f"ADF/HAADF candidates: {len(candidates):,}")
print(f"Sampled rows: {len(sampled_rows):,}")


In [ ]:
def signal_data(signal):
    if isinstance(signal, dict):
        return signal.get("data")
    return signal.data


def load_dm_image(path: Path) -> np.ndarray:
    signals = file_reader(str(path), lazy=False)
    if not isinstance(signals, list):
        signals = [signals]

    arrays = []
    for signal in signals:
        data = np.asarray(signal_data(signal))
        data = np.squeeze(data)
        if data.ndim >= 2:
            arrays.append(data)

    if not arrays:
        raise ValueError(f"No 2D image data found in {path}")

    image = arrays[0]
    while image.ndim > 2:
        image = image[0]
    if np.iscomplexobj(image):
        image = np.abs(image)
    return image.astype(np.float32, copy=False)


def display_scaled(image: np.ndarray) -> np.ndarray:
    finite = image[np.isfinite(image)]
    if finite.size == 0:
        return np.zeros_like(image, dtype=np.float32)

    low, high = np.percentile(finite, [1, 99.5])
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        low = float(np.nanmin(finite))
        high = float(np.nanmax(finite))
    if high <= low:
        return np.zeros_like(image, dtype=np.float32)

    return np.clip((image - low) / (high - low), 0, 1)


def title_for_row(row: sqlite3.Row) -> str:
    mag = row["formatted_magnification_text"] or ""
    date = row["acquisition_date"] or ""
    shape = row["data_shape"] or ""
    parts = [part for part in (mag, shape, date) if part]
    return " | ".join(parts)


In [ ]:
fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(10, 16), constrained_layout=True)
axes = np.asarray(axes).ravel()

failed = []
for ax, row in zip(axes, sampled_rows):
    source_path = ARCHIVE_ROOT / row["path"]
    try:
        image = load_dm_image(source_path)
        ax.imshow(display_scaled(image), cmap="gray", interpolation="nearest")
        ax.set_title(title_for_row(row), fontsize=6)
    except Exception as exc:
        failed.append((row["path"], repr(exc)))
        ax.text(0.5, 0.5, "load failed", ha="center", va="center", fontsize=8)
        ax.set_title(row["data_shape"] or "", fontsize=6)
    ax.set_axis_off()

for ax in axes[len(sampled_rows):]:
    ax.set_axis_off()

FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_PATH, dpi=300, bbox_inches="tight")
print(f"Saved {FIGURE_PATH}")
print(f"Failed loads: {len(failed)}")
failed[:5]


In [ ]:
# The sampled metadata rows are kept here for traceability.
[dict(row) for row in sampled_rows]
